In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()


# Hyperopt


Hyperopt is used in ML to optimize hyperparameters 





## Load Dataset

Let's load the clean Airbnb dataset in again , but this time we will split in 3, so we have a validation set as well as a training and test sets


In [2]:
file_path = f"/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, val_df, test_df = airbnb_df.randomSplit([.6, .2, .2], seed=42)


In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

#Extract, prepare and index Categorical Columns
categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]
string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")

#Extract numeric columns except the label
numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price"))]

#Join all columns together and vector assemble them
assembler_inputs = index_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

#Create RandomForest Regressor
rf = RandomForestRegressor(labelCol="price", maxBins=250, seed=42)

#Create Pipeline
pipeline = Pipeline(stages=[string_indexer, vec_assembler, rf])

#Create Regression Evauator
regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price")




## Hyperopt 

###Define *Objective Function*


First, define our **training objective function**. The objective function has two primary requirements:

1. An **input** **`params`** including hyperparameter values to use when training the model (`max_depth` & `num_trees`)
2. An **output** containing a loss metric on which to optimize, we will use `rmse`

We can copy the pipeline and updating some of the parameters




In [4]:
def objective_function(params):    
    
    # set the hyperparameters that we want to tune
    max_depth = params["max_depth"]
    num_trees = params["num_trees"]

    estimator = pipeline.copy({rf.maxDepth: max_depth, rf.numTrees: num_trees})
    model = estimator.fit(train_df)

    preds = model.transform(val_df)
    rmse = regression_evaluator.evaluate(preds)

    return rmse



Create a search space for

* max_depth: min=2, max=10, q=1
* num_trees: min=10, max=200, q=1

**This will fail if you are not in a ML cluster**


In [5]:
from hyperopt import hp

#Defining search space
#hp.quniform("name", min, max, q),

search_space = {
    "max_depth": hp.quniform("max_depth", 2, 10, 1),
    "num_trees": hp.quniform("num_trees", 10, 200, 1)
}







**`fmin()`** generates new hyperparameter configurations to use for your **`objective_function`**. It will evaluate as many models as max_evals parameters, using the information from the previous models to make a more informative decision for the next hyperparameter to try. 



In [6]:
from hyperopt import fmin, tpe, Trials
import numpy as np

num_evals = 5
trials = Trials()
best_hyperparam = fmin(fn=objective_function, 
                       space=search_space,
                       algo=tpe.suggest, 
                       max_evals=num_evals,
                       trials=trials,
                       rstate=np.random.default_rng(42))

# Retrain model on train & validation dataset and evaluate on test dataset
best_max_depth = best_hyperparam["max_depth"]
best_num_trees = best_hyperparam["num_trees"]
print(f'best_max_depth: {best_max_depth}')
print(f'best_num_trees: {best_num_trees}')

estimator = pipeline.copy({rf.maxDepth: best_max_depth, rf.numTrees: best_num_trees})
combined_df = train_df.union(val_df) # Combine train & validation together

pipeline_model = estimator.fit(combined_df)
pred_df = pipeline_model.transform(test_df)
rmse = regression_evaluator.evaluate(pred_df)


100%|██████████| 5/5 [00:18<00:00,  3.76s/trial, best loss: 39.27645925609726]
best_max_depth: 7.0
best_num_trees: 165.0
